# Gemini Crypto Data Downloader

Downloads public Gemini candle data and stores it by symbol under `./data/<symbol>/`.

In [7]:
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


In [8]:
# ---- Config ----
BASE_URL = "https://api.gemini.com"
SYMBOLS = ["btcusd", "ethusd", "solusd"]  # Gemini symbols are lowercase
TIMEFRAMES = ["1m", "5m", "1h", "1day"]
OUTPUT_ROOT = Path("./data")
SAVE_PARQUET = True
REQUEST_TIMEOUT = 20


In [ ]:
import time
import csv
from pathlib import Path
from typing import Optional, List
import requests

BASE_URL = "https://api.gemini.com"
# BASE_URL = "https://api.sandbox.gemini.com"

TF_MS = {
    "1m": 60_000,
    "5m": 5 * 60_000,
    "15m": 15 * 60_000,
    "30m": 30 * 60_000,
    "1h": 60 * 60_000,
    "6h": 6 * 60 * 60_000,
    "1d": 24 * 60 * 60_000,
}

def gemini_get_json(session: requests.Session, path: str, params=None, max_retries: int = 8):
    url = BASE_URL + path
    backoff = 0.5
    last_exc = None
    for _ in range(max_retries):
        try:
            r = session.get(url, params=params or {}, timeout=30)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(backoff)
                backoff = min(backoff * 2, 10.0)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            time.sleep(backoff)
            backoff = min(backoff * 2, 10.0)
    raise RuntimeError(f"Failed GET {url}") from last_exc

def list_symbols(session: requests.Session) -> List[str]:
    return gemini_get_json(session, "/v1/symbols")

def fetch_candles(session: requests.Session, symbol: str, timeframe: str):
    return gemini_get_json(session, f"/v2/candles/{symbol}/{timeframe}")

def latest_closed_timestamp_ms(timeframe: str) -> int:
    tf = TF_MS[timeframe]
    now_ms = int(time.time() * 1000)
    return (now_ms // tf) * tf - tf  # last fully closed candle

def get_last_timestamp_ms(file_path: Path) -> int:
    if not file_path.exists() or file_path.stat().st_size == 0:
        return -1

    # fast tail-read
    with file_path.open("rb") as f:
        f.seek(0, 2)
        size = f.tell()
        chunk = min(size, 8192)
        f.seek(-chunk, 2)
        data = f.read(chunk)

    for line in reversed(data.splitlines()):
        line = line.strip()
        if not line or line.startswith(b"timestamp_ms"):
            continue
        parts = line.split(b",")
        try:
            return int(parts[0])
        except Exception:
            continue

    # fallback: full scan
    last_ts = -1
    with file_path.open("r", newline="") as ftxt:
        reader = csv.reader(ftxt)
        for row in reader:
            if not row or row[0] == "timestamp_ms":
                continue
            try:
                last_ts = int(row[0])
            except Exception:
                pass
    return last_ts

def append_rows(file_path: Path, rows: List[List[float]]):
    is_new = (not file_path.exists()) or file_path.stat().st_size == 0
    file_path.parent.mkdir(parents=True, exist_ok=True)  # does NOT create a new “structure”, just ensures it exists
    with file_path.open("a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["timestamp_ms", "open", "high", "low", "close", "volume"])
        w.writerows(rows)

def normalize_filter_sort(candles, min_ts_exclusive: int, max_ts_inclusive: int) -> List[List[float]]:
    out = []
    for row in candles:
        if not isinstance(row, (list, tuple)) or len(row) < 6:
            continue
        ts = int(row[0])
        if ts > min_ts_exclusive and ts <= max_ts_inclusive:
            out.append([ts, float(row[1]), float(row[2]), float(row[3]), float(row[4]), float(row[5])])
    out.sort(key=lambda x: x[0])
    return out

def update_symbol_file(session: requests.Session, symbol: str, timeframe: str, out_dir: Path, sleep_between: float = 0.02) -> int:
    file_path = out_dir / f"{symbol}.data"
    last_ts = get_last_timestamp_ms(file_path)
    closed_ts = latest_closed_timestamp_ms(timeframe)

    if last_ts >= closed_ts:
        return 0  # already up to date

    candles = fetch_candles(session, symbol, timeframe)
    new_rows = normalize_filter_sort(candles, min_ts_exclusive=last_ts, max_ts_inclusive=closed_ts)

    if new_rows:
        append_rows(file_path, new_rows)

    time.sleep(sleep_between)
    return len(new_rows)

def update_all_symbols_inplace(
    timeframe: str,
    out_dir: str,
    max_symbols: Optional[int] = None,
    sleep_between: float = 0.02,
):
    if timeframe not in TF_MS:
        raise ValueError(f"Unsupported timeframe {timeframe}. Choose from {list(TF_MS.keys())}")

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)  # ensures your existing folder exists

    with requests.Session() as session:
        symbols = ["btcusd", "ethusd", "solusd"]  # <-- your exact list of symbols
        # symbols = list_symbols(session)
        if max_symbols is not None:
            symbols = symbols[:max_symbols]

        # keep a record of what symbols you used
        (out_path / "symbols.txt").write_text("\n".join(symbols))

        total_new = 0
        for i, sym in enumerate(symbols, 1):
            try:
                n_new = update_symbol_file(session, sym, timeframe, out_path, sleep_between=sleep_between)
                total_new += n_new
                if i % 50 == 0 or i == len(symbols):
                    print(f"[{i}/{len(symbols)}] appended_rows_total={total_new}")
            except Exception as e:
                print(f"[{i}/{len(symbols)}] FAILED {sym}: {type(e).__name__}: {e}")

    print(f"Done. Updated files in-place at: {out_path.resolve()}")

if __name__ == "__main__":
    update_all_symbols_inplace(
        timeframe="1m",
        out_dir=".data/gemini/ohlcv_1m_7d",  # <-- your exact directory
        max_symbols=None,                   # or set 50 to test
        sleep_between=0.02
    )

[50/451] appended_rows_total=71999
[100/451] appended_rows_total=143999
[127/451] FAILED efilfil: RuntimeError: Failed GET https://api.gemini.com/v2/candles/efilfil/1m
[150/451] appended_rows_total=214559
[193/451] FAILED gusdusd: RuntimeError: Failed GET https://api.gemini.com/v2/candles/gusdusd/1m
[200/451] appended_rows_total=285119
[250/451] appended_rows_total=357116
[300/451] appended_rows_total=429116
[350/451] appended_rows_total=501116
[400/451] appended_rows_total=573116
[450/451] appended_rows_total=645116
[451/451] appended_rows_total=646556
Done. Updated files in-place at: /home/ec2-user/gemini_comp/.data/gemini/ohlcv_1m_7d
